In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/support_messages.csv")
df.head()

,message_id,received_at,channel,message_text,language_hint,customer_tier,intent,resolved_minutes
0,MSG-001549,2026-05-15 20:21:00,whatsapp,bonjour the transfer ispending since yesterday,en,basic,failed_transfer,32
1,MSG-013890,23/02/2026 23:00,sms,hello money commot for ma account but e no lan...,mixed,plus,failed_transfer,88
2,MSG-025260,2026-05-20 23:36:00,sms,abeg l'application ne s'ouvre pas pls,fr,plus,app_technical,143
3,MSG-009296,2026-03-13 10:52:00,whatsapp,pls money commot for ma account but e no land,pi,plus,failed_transfer,88
4,MSG-010671,2025-10-13 12:29:00,sms,pls the app will not pen,en,basic,app_technical,103


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44600 entries, 0 to 44599
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   message_id         44600 non-null  str  
 1   received_at        44600 non-null  str  
 2   channel            44600 non-null  str  
 3   message_text       44600 non-null  str  
 4   language_hint      44600 non-null  str  
 5   customer_tier      44600 non-null  str  
 6   intent             44600 non-null  str  
 7   resolved_minutes   44600 non-null  int64
 8   normalized_intent  44600 non-null  str  
dtypes: int64(1), str(8)
memory usage: 8.9 MB


In [3]:
def normalizer(text: str) -> str:
    text = text.lower().strip()
    text = " ".join(text.split())

    return text

In [13]:
df["normalized_intent"] = df["message_text"].map(normalizer)
df.head()

,message_id,received_at,channel,message_text,language_hint,customer_tier,intent,resolved_minutes,normalized_intent
0,MSG-001549,2026-05-15 20:21:00,whatsapp,bonjour the transfer ispending since yesterday,en,basic,failed_transfer,32,bonjour the transfer ispending since yesterday
1,MSG-013890,23/02/2026 23:00,sms,hello money commot for ma account but e no lan...,mixed,plus,failed_transfer,88,hello money commot for ma account but e no lan...
2,MSG-025260,2026-05-20 23:36:00,sms,abeg l'application ne s'ouvre pas pls,fr,plus,app_technical,143,abeg l'application ne s'ouvre pas pls
3,MSG-009296,2026-03-13 10:52:00,whatsapp,pls money commot for ma account but e no land,pi,plus,failed_transfer,88,pls money commot for ma account but e no land
4,MSG-010671,2025-10-13 12:29:00,sms,pls the app will not pen,en,basic,app_technical,103,pls the app will not pen


In [15]:
duplicates = df[df.duplicated("normalized_intent", keep=False)]
duplicates.head()


,message_id,received_at,channel,message_text,language_hint,customer_tier,intent,resolved_minutes,normalized_intent
2,MSG-025260,2026-05-20 23:36:00,sms,abeg l'application ne s'ouvre pas pls,fr,plus,app_technical,143,abeg l'application ne s'ouvre pas pls
3,MSG-009296,2026-03-13 10:52:00,whatsapp,pls money commot for ma account but e no land,pi,plus,failed_transfer,88,pls money commot for ma account but e no land
5,MSG-029693,2026-01-25 15:19:00,whatsapp,bonjour money left my account but never reached 🙏,en,premium,tariff_question,30,bonjour money left my account but never reached 🙏
6,MSG-002561,2026-02-01 06:23:00,whatsapp,hi envoyez moi mon solde svp asap,fr,basic,balance_enquiry,36,hi envoyez moi mon solde svp asap
8,MSG-025254,2025-12-11 12:53:00,whatsapp,HI J'AI ENVOYE DE L'ARGENT ET CE N'EST PAS ARR...,fr,plus,failed_transfer,1,hi j'ai envoye de l'argent et ce n'est pas arr...


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2
)

x= vectorizer.fit_transform(df["normalized_intent"])

nn = NearestNeighbors(
    n_neighbors=2,
    metric="cosine"
)

nn.fit(x)

distances, indices = nn.kneighbors(x)
similarities = 1 - distances[:, 1]

In [18]:
df["nearest_similarity"] = similarities
candidates = df[df["nearest_similarity"] >= 0.90]

print(len(candidates))

36184


In [32]:
df.tail()

,message_id,received_at,channel,message_text,language_hint,customer_tier,intent,resolved_minutes,normalized_intent,nearest_similarity
44595,MSG-011843,06/10/2025 15:54,app chat,good morning how much i get for ma account ver...,mixed,plus,balance_enquiry,11,good morning how much i get for ma account ver...,1.000000
44596,MSG-001805,2026-01-17 10:25:00,whatsapp,hi the app says network error pls,en,plus,app_technical,50,hi the app says network error pls,1.000000
44597,MSG-002035,2025-12-12 18:46:00,whatsapp,pls abeg check ma balance check my balance pls...,mixed,basic,balance_enquiry,58,pls abeg check ma balance check my balance pls...,0.909372
44598,MSG-035244,2026-02-09 18:20:00,whatsapp,hello how much be di charge for send money que...,mixed,basic,tariff_question,155,hello how much be di charge for send money que...,1.000000
44599,MSG-013719,2026-01-10 10:18:00,app chat,je n'ai pas autorise cette transaction asap,fr,basic,fraud_report,224,je n'ai pas autorise cette transaction asap,0.948731


In [34]:
pairs = []

for i in range(len(df)):
    for j in indices[i]:
        if i != j:
            similarity = 1 - distances[i][list(indices[i]).index(j)]

            if similarity >= 0.90:
                pairs.append((i, j))

In [36]:
import networkx as nx

G = nx.Graph()
G.add_nodes_from(range(len(df)))

for i, j in pairs:
    G.add_edge(i, j)

groups = list(nx.connected_components(G))

In [37]:
df["duplicate_group"] = -1

for group_id, group in enumerate(groups):
    for idx in group:
        df.loc[idx, "duplicate_group"] = group_id

In [38]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, temp_idx = next(
    splitter.split(
        df,
        groups=df["duplicate_group"]
    )
)

train = df.iloc[train_idx]
temp = df.iloc[temp_idx]

In [39]:
temp

,message_id,received_at,channel,message_text,language_hint,customer_tier,intent,resolved_minutes,normalized_intent,nearest_similarity,duplicate_group
0,MSG-001549,2026-05-15 20:21:00,whatsapp,bonjour the transfer ispending since yesterday,en,basic,failed_transfer,32,bonjour the transfer ispending since yesterday,0.850978,0
3,MSG-009296,2026-03-13 10:52:00,whatsapp,pls money commot for ma account but e no land,pi,plus,failed_transfer,88,pls money commot for ma account but e no land,1.000000,3
8,MSG-025254,2025-12-11 12:53:00,whatsapp,HI J'AI ENVOYE DE L'ARGENT ET CE N'EST PAS ARR...,fr,plus,failed_transfer,1,hi j'ai envoye de l'argent et ce n'est pas arr...,1.000000,8
14,MSG-010797,2025-10-06 02:47:00,app chat,PLS Y A T IL DES FRAIS DE RETRAIT DEM DEY CHAR...,mixed,premium,tariff_question,52,pls y a t il des frais de retrait dem dey char...,1.000000,14
15,MSG-014510,2025-11-03 21:25:00,whatsapp,how much i get for my account 🙏,en,basic,tariff_question,22,how much i get for my account 🙏,1.000000,15
...,...,...,...,...,...,...,...,...,...,...,...
44574,MSG-005741,2025-11-26 19:45:00,call centre note,abeg lock my pin and give new one,en,basic,pin_reset,130,abeg lock my pin and give new one,0.866951,16199
44575,MSG-021023,2026-02-21 20:39:00,facebook,bonjou app dey crash for ma phone,pi,premium,app_technical,122,bonjou app dey crash for ma phone,0.935539,8296
44576,MSG-026872,2025-11-30 09:55:00,whatsapp,una help me,fr,plus,app_technical,153,una help me,1.000000,127
44577,MSG-038959,2026-05-11 10:17:00,sms,good morning excellent service today abeg,en,basic,praise,95,good morning excellent service today abeg,0.933276,9975
